# Lab 10: Scalar Backpropagation

**Course:** Deep Learning for Computer Vision — 2026  
**Lecture connection:** Lecture 6

Implement a scalar computational graph, compute its gradients by hand, and compare them with numerical gradients.

## 1. Setup

Keep the notebook and `scalar_backprop.py` together. In Google Colab, upload the complete `Lab10` folder before opening the notebook. Run the setup cell again after editing `scalar_backprop.py`.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib
import sys

import torch

candidate_dirs = [
    Path.cwd(),
    Path.cwd() / 'Lab10',
    Path.cwd() / 'DNN' / 'Lab10',
    Path.cwd() / 'Labs' / 'DNN' / 'Lab10',
    Path.cwd() / 'Deep-Learning-for-Computer-Vision' / 'Labs' / 'DNN' / 'Lab10',
]
lab_dir = next(
    (path.resolve() for path in candidate_dirs if (path / 'scalar_backprop.py').is_file()),
    None,
)
if lab_dir is None:
    raise FileNotFoundError(
        'Could not find scalar_backprop.py. Keep it together with the notebook.'
    )

if str(lab_dir) not in sys.path:
    sys.path.insert(0, str(lab_dir))
sys.modules.pop('scalar_backprop', None)

import scalar_backprop

importlib.reload(scalar_backprop)
torch.set_default_dtype(torch.float64)

print('PyTorch:', torch.__version__)
print('Lab directory:', lab_dir)
print('Implementation:', Path(scalar_backprop.__file__).resolve())


PyTorch: 2.14.0+cpu
Lab directory: C:\Users\Alina\OneDrive\Desktop\СV\Deep-Learning-for-Computer-Vision\Labs\DNN\Lab10
Implementation: C:\Users\Alina\OneDrive\Desktop\СV\Deep-Learning-for-Computer-Vision\Labs\DNN\Lab10\scalar_backprop.py


## 2. The scalar graph

The graph from the lecture is:

\[
s_0 = w_0x_0, \quad s_1 = w_1x_1, \quad s_2 = s_0+s_1,
\]

\[
s_3 = s_2+w_2, \quad L = \operatorname{sigmoid}(s_3).
\]

Every value is a scalar tensor. The `cache` returned by the forward pass stores the intermediate values needed in the backward pass.

In [2]:
values = scalar_backprop.make_fixture()
for name, value in values.items():
    print(f'{name} = {value.item():+.1f}')


w0 = +2.0
x0 = -1.0
w1 = -3.0
x1 = -2.0
w2 = -3.0


## 3. Exercise 1 — Forward pass

Complete `sigmoid` and `scalar_forward` in `scalar_backprop.py`. The fixed inputs should give:

\[
s_0=-2, \quad s_1=6, \quad s_2=4, \quad s_3=1, \quad L \approx 0.7311.
\]

In [3]:
importlib.reload(scalar_backprop)

loss, cache = scalar_backprop.scalar_forward(values)
for name in ('s0', 's1', 's2', 's3', 'loss'):
    print(f'{name}: {cache[name].item():+.6f}')

assert torch.allclose(cache['s0'], torch.tensor(-2.0))
assert torch.allclose(cache['s1'], torch.tensor(6.0))
assert torch.allclose(cache['s2'], torch.tensor(4.0))
assert torch.allclose(cache['s3'], torch.tensor(1.0))
assert torch.allclose(loss, torch.tensor(0.7310585786300049))
print('Forward-pass check passed.')


s0: -2.000000
s1: +6.000000
s2: +4.000000
s3: +1.000000
loss: +0.731059
Forward-pass check passed.


## 4. Exercise 2 — Manual backward pass

Complete `scalar_backward`. Start with the gradient 1 at `loss`. Move backwards through sigmoid, the two additions, and the two multiplication gates. Do not call `backward` or `torch.autograd` inside `scalar_backward`.

In [4]:
importlib.reload(scalar_backprop)

loss, cache = scalar_backprop.scalar_forward(values)
grads = scalar_backprop.scalar_backward(cache)
for name, gradient in grads.items():
    print(f'dL/d{name}: {gradient.item():+.6f}')

expected_grads = {
    'w0': torch.tensor(-0.19661193324148185),
    'x0': torch.tensor(0.3932238664829637),
    'w1': torch.tensor(-0.3932238664829637),
    'x1': torch.tensor(-0.5898357997244456),
    'w2': torch.tensor(0.19661193324148185),
}
for name, expected in expected_grads.items():
    assert torch.allclose(grads[name], expected)
print('Manual-backpropagation check passed.')


dL/dw2: +0.196612
dL/dw0: -0.196612
dL/dx0: +0.393224
dL/dw1: -0.393224
dL/dx1: -0.589836
Manual-backpropagation check passed.


## 5. Exercise 3 — Numerical gradient check

A centered finite difference estimates a derivative by slightly increasing and decreasing one input. The numerical and manual gradients should agree to a very small relative error.

In [5]:
importlib.reload(scalar_backprop)

report = scalar_backprop.gradient_report(values)
for name, (numerical, analytic, error) in report.items():
    print(
        f'{name}: numerical={numerical.item():+.8f}  ' 
        f'analytic={analytic.item():+.8f}  relative_error={error:.3e}'
    )

maximum_error = max(error for _, _, error in report.values())
print('maximum relative error:', maximum_error)
assert maximum_error < 1e-7


w0: numerical=-0.19661193  analytic=-0.19661193  relative_error=1.860e-11
x0: numerical=+0.39322387  analytic=+0.39322387  relative_error=1.860e-11
w1: numerical=-0.39322387  analytic=-0.39322387  relative_error=1.860e-11
x1: numerical=-0.58983580  analytic=-0.58983580  relative_error=1.860e-11
w2: numerical=+0.19661193  analytic=+0.19661193  relative_error=1.860e-11
maximum relative error: 1.8595106975625306e-11


## 6. Compare with PyTorch automatic differentiation

PyTorch can build the same graph and compute gradients automatically. This check confirms that the manual gradients match PyTorch's result.

In [6]:
importlib.reload(scalar_backprop)

autograd_values = {
    name: value.detach().clone().requires_grad_(True)
    for name, value in values.items()
}
reference_loss, _ = scalar_backprop.scalar_forward(autograd_values)
reference_loss.backward()

_, cache = scalar_backprop.scalar_forward(values)
manual_grads = scalar_backprop.scalar_backward(cache)
for name in manual_grads:
    autograd_gradient = autograd_values[name].grad
    print(
        f'{name}: manual={manual_grads[name].item():+.8f}  ' 
        f'autograd={autograd_gradient.item():+.8f}'
    )
    assert torch.allclose(manual_grads[name], autograd_gradient)
print('Autograd comparison passed.')


w2: manual=+0.19661193  autograd=+0.19661193
w0: manual=-0.19661193  autograd=-0.19661193
x0: manual=+0.39322387  autograd=+0.39322387
w1: manual=-0.39322387  autograd=-0.39322387
x1: manual=-0.58983580  autograd=-0.58983580
Autograd comparison passed.


## 7. Wrap-up

Answer briefly in a new Markdown cell:

1. Why does an addition gate copy its upstream gradient to both inputs?

    Because the derivative of a sum with respect to any of its terms is exactly 1. By the chain rule, multiplying the upstream gradient by 1 simply passes (copies) the gradient unchanged to both inputs.

2. For a multiplication gate, why does the gradient for one input use the other input?

    The derivative of a product with respect to one factor is the other factor. By the chain rule, the gradient flowing back to a must be scaled by the local derivative, which is the value of b.

3. What does a small numerical-gradient error tell you about a manual backward pass?

    It tells you that your manual application of the chain rule (the analytic gradient) is correct. The small difference is just floating-point rounding, meaning the backpropagation logic matches the actual slope of the forward function.